# Day Trader Metrics

No API keys needed in this, no trades are actioned, only an analysis of public market data.

1. Data sourced live using `CCXT` API library, no scraping required.
2. Indicators derived using `pandas-ta-classic`, replacing `numpy.2`



### Environment Setup

Run the following cell each time a new kernel starts, or whenever you start a new session. To avoid setup issues across different users, the `venv` environment was removed but package list requirements.txt list was maintained. This cell builds everything again and configures the notebook graphics. 

Make sure to log into Binance and Alpaca accounts before running this setup below. To avoid api issues, the prferred Binance.US login is accessed here: https://docs.ccxt.com/en/latest/exchange-markets.html


In [ ]:
import sys, subprocess
from pathlib import Path

# assign Python kernel,
REQUIRED_PY = (3, 11)
if sys.version_info[:2] != REQUIRED_PY: raise RuntimeError(
        f"This notebook needs Python {REQUIRED_PY[0]}.{REQUIRED_PY[1]}, "
        f"but the kernel is {sys.version.split()[0]}. Switch the kernel and re-run.")

# Install packages & versions in requirements.txt
REQUIREMENTS = Path("inputs/requirements.txt")
if not REQUIREMENTS.exists():
    raise FileNotFoundError(f"{REQUIREMENTS} not found — run from the repo root.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
     "--break-system-packages", "--disable-pip-version-check",
     "-r", str(REQUIREMENTS)], check=True,)

# import package modules and naming
import warnings, numpy as np, pandas as pd
from datetime import datetime
import ccxt
import pandas_ta_classic as ta
from pykalman import KalmanFilter
import plotly, plotly.graph_objects as go, plotly.io as pio
warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"  

# quick sanity check
print(f"Environment ready  ·  python {sys.version.split()[0]}  ({sys.executable})")
print(f"  ccxt {ccxt.__version__} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · plotly {plotly.__version__}")

Environment ready  ·  python 3.11.13  (/Users/seamus/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/bin/python)
  ccxt 4.5.59 · pandas 2.3.3 · numpy 2.4.6 · plotly 6.8.0


### Import Data

Live price data is sourced between 24 hours and 1 week from multiple trading platforms. Five filters were highlighted below, including `EXCHANGE`, `SANDBOX`, `TIMEFRAME`, `LIMIT`, and `TOP_N`. Currently, the`ccxt` library of APIs maintains access to 105 leading crypto [exchange platforms](https://github.com/ccxt/ccxt/wiki/Exchange-Markets). Happy browsing!

In [12]:
EXCHANGE  = "binance"   # 105 platforms available, see https://github.com/ccxt/ccxt/wiki/Exchange-Markets)
SANDBOX   = False       # True = testnet / fake money (mainly for the trading step later)
TIMEFRAME = "1d"        # candle size: "1m","5m","1h","4h","1d","1w"
LIMIT     = 300         # candles pulled per coin (max 1000)
TOP_N     = 20          # how many of the most-traded /USDT pairs to scan

exchange = getattr(ccxt, EXCHANGE)()
exchange.set_sandbox_mode(SANDBOX)
print(f"Configured: {EXCHANGE} | sandbox={SANDBOX} | {TIMEFRAME} candles x{LIMIT} | top {TOP_N}")

Configured: binance | sandbox=False | 1d candles x300 | top 20


In [ ]:
def calculate_indicator(symbol, timeframe=TIMEFRAME, limit=LIMIT):
    bars = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(bars[:-1], columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.set_index("timestamp")
    close, low = df["close"].iloc[-1], df["low"].iloc[-1]

    # Kalman threshol & smoother
    kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1],
                      initial_state_mean=0, initial_state_covariance=1,
                      observation_covariance=1, transition_covariance=.01)
    state_means, _ = kf.filter(df["close"].values)
    df["kf_mean"] = state_means
    kalman = df["kf_mean"].iloc[-1]
    above_kalman = bool(low > kalman)

    # Trend: Is EMA-14 leading the Kalman mean?
    df.ta.ema(length=14, append=True)
    ema_cross = bool(df["EMA_14"].iloc[-1] > kalman)

    # Bollinger(14) mean-reversion envelope.
    bb = df.ta.bbands(length=14)
    bbl, bbu = bb["BBL_14_2.0"].iloc[-1], bb["BBU_14_2.0"].iloc[-1]

    # Ichimoku projectionse.
    ich = df.ta.ichimoku()[1]
    isa_9, isb_26 = ich["ISA_9"].iloc[-1], ich["ISB_26"].iloc[-1]

    # Archer MA trend flag, RSI, Choppiness.
    amat = bool(df.ta.amat()["AMATe_LR_8_21_2"].iloc[-1] == 1)
    rsi = float(df.ta.rsi().iloc[-1])
    chop = round(float(df.ta.chop().iloc[-1]), 2)
    
    # Candle shape check: any doji / dragonfly / gravestone candles?
    o, h, l, c = df["open"].iloc[-1], df["high"].iloc[-1], df["low"].iloc[-1], df["close"].iloc[-1]
    rng   = h - l
    body  = abs(c - o)
    upper = h - max(o, c)
    lower = min(o, c) - l
    doji       = bool(rng > 0 and body <= 0.10 * rng)
    dragonfly  = bool(doji and lower >= 0.6 * rng and upper <= 0.10 * rng)
    gravestone = bool(doji and upper >= 0.6 * rng and lower <= 0.10 * rng)

    # MACD (12,26,9)
    macd_line   = df["close"].ewm(span=12, adjust=False).mean() - df["close"].ewm(span=26, adjust=False).mean()
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd, sig           = macd_line.iloc[-1], signal_line.iloc[-1]
    macd_prev, sig_prev = macd_line.iloc[-2], signal_line.iloc[-2]
    macd_buy  = bool(macd_prev <= sig_prev and macd > sig)
    macd_sell = bool(macd_prev >= sig_prev and macd < sig)
    macd_below_zero = bool(macd < 0)
    buy  = amat and ema_cross and above_kalman
    sell = (not amat) and (not ema_cross) and (not above_kalman)    
    row = dict(Symbol=symbol, Buy=buy, Sell=sell, Close=round(float(close), 4),
               RSI=round(rsi, 2), Chop=chop, AMAT=amat,
               Ichimoku_9=round(float(isa_9), 4), Ichimoku_26=round(float(isb_26), 4),
               EMA_gt_Kalman=ema_cross, Low_gt_Kalman=above_kalman,
               Doji=doji, Dragonfly=dragonfly, Gravestone=gravestone,
               MACD=round(float(macd),4), MACD_Signal=round(float(sig),4),
               MACD_Buy=macd_buy, MACD_Sell=macd_sell, MACD_below_zero=macd_below_zero)
               
    return df, row

def plot(symbol):
    df, _ = calculate_indicator(symbol)
    fig = go.Figure(go.Candlestick(x=df.index, open=df.open, high=df.high, low=df.low, close=df.close, name=symbol))
    fig.add_trace(go.Scatter(x=df.index, y=df["kf_mean"], name="Kalman", line=dict(color="orange", width=2), opacity=0.7))
    fig.add_trace(go.Scatter(x=df.index, y=df["EMA_14"], name="EMA-14", line=dict(color="purple", width=2), opacity=0.7))
    fig.update_layout(title=symbol, xaxis_rangeslider_visible=False)
    return fig

def color_boolean(val):
    if val is True:  return "background-color: lightgreen"
    if val is False: return "background-color: pink"
    return "background-color: lightblue"

print("Engine ready.")

### Analyze Data

The following pulls 300 daily candles and computes the following:

- Kalman Filter: a price smoothing noisy fluctuations, key indicator of close of day price strength
- Bollinger Bands: An elastic envelope derived from 14-day price average. Narrowing suggests quiet market, flaring suggest volatile.
- Ichimoku Spans: Two mean price projections used as trend map, suggesting uptrend or downtrends between 9 adn 26 days.
- AMAT: Archer Moving Average Trends used as binary indicators showing diverging fast and slow averages, or aligning showing sustained trend (1 = "trending up").
- Relative Strength Index (RSI). Speedometer, above 70, price running fast risks overselling and then cooling off. Below 30 may mean due to bounce.
- Choppiness Index: High values suggest inertia, low suggest clean trend.

Key values in these metrics in sell and buy point variables above, copied here again for review: `ema_crossover = ema_14 > ema_91`

The originals scanned every Binance.US ticker, which is slow and noisy. Here we take the most liquid spot /USDT pairs by quote volume. Raise `TOP_N` to widen the scan; each extra symbol adds roughly a second.


In [ ]:
tickers = exchange.fetch_tickers()
pairs = [(s, d.get('quoteVolume') or 0) for s, d in tickers.items()
         if s.endswith('/USDT') and ':' not in s]
universe = [s for s, _ in sorted(pairs, key=lambda x: -x[1])[:TOP_N]]
today = datetime.now().strftime('%Y-%m-%d')
print(f"{today}: scanning {len(universe)} pairs")
universe

### Rank Data

Run the engine over the universe into one table. The `try/except` skips symbols too new to have full indicator history. For example a token listed days ago has no Ichimoku cloud and skips these values instead of hiding them. 

In [15]:
rows = []
for symbol in universe:
    try: rows.append(calculate_indicator(symbol)[1])
    except Exception as e: print(f"skip {symbol}: {type(e).__name__}")
results = pd.DataFrame(rows)
print(f"\nscored {len(results)} pairs | {int(results.Buy.sum())} buys | {int(results.Sell.sum())} sells")
results.head()

skip RE/USDT: IndexError
skip MEGA/USDT: TypeError
skip SPCXB/USDT: KeyError

scored 17 pairs | 4 buys | 2 sells


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,USDC/USDT,False,False,1.0006,52.78,17.31,True,1.0080,1.0108,False,False,False,False,False,-0.0000,-0.0000,False,False,True
1,BTC/USDT,False,False,64509.4000,38.74,54.00,True,66314.5150,70990.4550,False,False,False,False,False,-2449.8918,-3053.8891,False,False,True
2,ETH/USDT,False,False,1750.6000,41.55,51.49,True,1777.1525,1964.7100,False,False,False,False,False,-87.2995,-112.4537,False,False,True
3,WLD/USDT,True,False,0.6587,68.50,54.27,True,0.5336,0.4748,True,True,False,False,False,0.0745,0.0597,False,False,False
4,USD1/USDT,True,False,1.0008,62.70,63.37,True,1.0001,1.0000,True,True,False,False,False,0.0002,0.0001,False,False,False


## Performance Metrics

### MACD Trends
The metric of Moving Average Convergence Divergence is monitored as an indcator of changing momentum, which is quanitified by the velocity of acceleration or decay in price trends ([Appel, 1985](https://books.google.ca/books/about/The_Moving_Average_Convergence_divergenc.html?id=xg9LAQAACAAJ&redir_esc=y). Because MACD is quantitatively faster than the units of EMA, these lines are expected to cross at the same moment the 12-EMA crosses the 26-EMA line. 
The nature of these `crossovers` are focus of day trading with buy points selected
when MACD crosses above zero line, while sell points selected as it crosses below. 
Goes without saying, crossing above the zero line signals bullish trend, 
while passing below zero signals bearish trend. Tricky thing is while its earlier signals 
offer higher potential than the zero line, it carries risk of false starts.

Beneath the line graph, the histogram can add to our predictions providing distances
between lines that guide us to whenm lines cross. The height of these distances represents 
momentum, so that a growing histogram indicates accelerating divergence. Alternatively, a
shrinking histogram of converging lines suggests momentum is fading signalling a likely turn.

Divergence presents important metric in day trading that is least mechanical to read. 
Observing changing swings between MACD highs and lows, we can identify a 
shift in trends is expected. Bearish divergence can be characterised by 
higher highs while MACD show lower highs that caution an uptrend is
hollow. aternatively, a bullish divergence can appear with lower lows while MACD presents 
higher lows that caution a downtrend is exhausting. Need guidance on this, but seems  
divergence is mostly used to protect a position before change, rather than a precise entry to trade.

## Diverge/Convergence Quadrants
The full taxonomy compares the last two price swings against the matching MACD
swings. Notation: HH higher high, HL higher low, LH lower high, LL lower low.

| Price | MACD | Source term | Standard term | Reading | Action |
|---|---|---|---|---|---|
| Higher high | Lower high | Divergence | Regular bearish | reversal / retracement | SELL |
| Lower high | Higher high | Convergence | Hidden bearish | trend continuation (down) | SELL |
| Higher low | Lower low | Divergence | Hidden bullish | trend continuation (up) | BUY |
| Lower low | Higher low | Convergence | Regular bullish | reversal / retracement | BUY |

![divergence quadrants](/outputs/macd_lab/divergence_matrix.png)

Two cautions from the research. First, terminology is not standardised: GoodCrypto
defines "divergence" as price stronger than the indicator and "convergence" as
price weaker, which is the geometry above but the opposite word to how many texts
use them. Read the HH/HL/LH/LL pattern, not the label. Second, a single chart can
show mixed signals: in their BTC example the peaks diverged while the troughs
converged, netting out bearish only because the peak structure dominated. Weigh
both swing series, do not act on one line in isolation.

### MACD of Crypto

MACD is well respected in BTC and ETH markets, but cannot be handled in isolation. It is recommended  
to consider alonside position sizing and a expected stop. Some suggest a good crypto 
technique  should draw the trendline on the MACD itself, not only on price. For example, 
in the May 2021 BTC top, price held a rising trendline while the MACD was already 
tracing a falling one. The MACD trendline broke first. The same setup appeared on the 
SOL 4h chart, pushing higher highs into resistance while MACD showed clearly lower highs. Was this 
a textbook bearish divergence, as it preceded the stall?

Timeline is important, but does not lessen the MACD metric; a 4h or 1h
MACD updates faster than a daily one. The real limiter to watch out for is 
volatility: the more violent the asset the risker its forecast, hence 
reliance on corroboration or triangulation in fast markets.


## Corroborating Metrics
MACD gives momentum but poor at signaling overbought/oversold thresholds, so traders may 
confirm crossover against a second indicator and only act when both agree. In rough order 
of their selective applications (CITE):

| Partner | What it adds | Entry rule | Exit |
|---|---|---|---|
| Relative Vigor Index | closing strength vs range | both cross same direction | MACD opposite cross |
| Money Flow Index | price + volume, few signals | MFI overbought/oversold then MACD cross | MACD opposite cross |
| TEMA (50) | triple-smoothed trend | price breaks TEMA and MACD crosses | contrary signal from both |
| TRIX | momentum oscillator | MACD cross matched by TRIX zero-cross | MACD cross, or looser, TRIX zero-cross |
| Awesome Oscillator | 5/34 SMA momentum | MACD cross confirmed by AO | both turn contrary |
| MA (20) | trend validation | price tests the 20-MA, then MACD crosses up | — |

The shared idea is the guardrail we already build into the metric: do not take a
bare crossover, require corroboration. MFI's threshold, which provides good triangulation here, 
considered variable of interest in next iterations, paritcularly since histogram and slope already 
cover RVI, TRIX, and AO fields.

### Final MACD Metrics
The current iteration targets these metrics by defining the following variables, including 
`cross_up` and `cross_down` crossover points; as well as `guarded_buy` and `guarded_sell` 
constraints. These were applied as conservative measures around noise which we found 
recommended in majority of readings. Histogram variables were derived for both `hist_slope` 
and `converging` flags to highlight early fade nearing zero. Divergence points were constructed
at swing pivot points for variables of `bear_div` and `bull_div` representing  bearish and bullish 
quadrant data of regular bounds. MACD breaks and hidden divvergence, which are not yet coded, are 
next target. `strength` is compiled from band clearances, slope spikes, zero field, and 
corroboration by divergence, which was numerically viewed as significantly weighted swing series.

![BTC preview](/outputs/macd_lab/preview_btc.png)


### Entry Points
Buy trends are displayed first with Kalman flags rendered, then lowest RSI at the top. Sells are shown as the mirror, which the originals' automation checklist is built upon.

In [16]:
top = (results.sort_values(['Buy','EMA_gt_Kalman','AMAT','RSI'], ascending=[False,False,False,True]).reset_index(drop=True).head(10))
bottom = (results.sort_values(['Sell','AMAT','RSI'],ascending=[False,False,False]).reset_index(drop=True).head(10))
print("ranked.")

if top['Buy'].any():
    print("Entry candidates showing bullish trends and Kalman lead prices:")
    for s in top.loc[top['Buy'], 'Symbol']: print(" ", s)
else: print("No long signals showing strongest-ranked names.")

top.style.map(color_boolean)

ranked.
Entry candidates showing bullish trends and Kalman lead prices:
  XLM/USDT
  USD1/USDT
  XPL/USDT
  WLD/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,XLM/USDT,True,False,0.225600,61.690000,55.770000,True,0.210500,0.218900,True,True,False,False,False,0.007700,0.007300,True,False,False
1,USD1/USDT,True,False,1.000800,62.700000,63.370000,True,1.000100,1.000000,True,True,False,False,False,0.000200,0.000100,False,False,False
2,XPL/USDT,True,False,0.112700,67.440000,39.660000,True,0.091700,0.091700,True,True,False,False,False,0.001600,-0.002400,False,False,False
3,WLD/USDT,True,False,0.658700,68.500000,54.270000,True,0.533600,0.474800,True,True,False,False,False,0.074500,0.059700,False,False,False
4,NEAR/USDT,False,False,2.182000,51.310000,50.320000,True,2.348000,2.166000,True,False,False,False,False,0.054100,0.068300,False,False,False
5,BTC/USDT,False,False,64509.400000,38.740000,54.000000,True,66314.515000,70990.455000,False,False,False,False,False,-2449.891800,-3053.889100,False,False,True
6,ETH/USDT,False,False,1750.600000,41.550000,51.490000,True,1777.152500,1964.710000,False,False,False,False,False,-87.299500,-112.453700,False,False,True
7,XRP/USDT,False,False,1.186800,44.710000,49.170000,True,1.202100,1.299900,False,False,False,False,False,-0.037700,-0.051000,False,False,True
8,SOL/USDT,False,False,72.050000,46.200000,50.960000,True,71.515000,79.270000,False,False,False,False,False,-2.914300,-4.120800,False,False,True
9,ZEC/USDT,False,False,477.230000,48.930000,38.680000,True,470.817500,470.060000,False,False,False,False,False,-14.841500,-19.769500,False,False,True


A chart showing strongest buy candidate, with the Kalman and EMA-14 trend lines:

In [1]:
from IPython.display import HTML
HTML(plot(top['Symbol'].iloc[0]).to_html(include_plotlyjs="cdn", full_html=False))

NameError: name 'plot' is not defined

In [18]:
# Save the chart as a standalone, self-contained HTML file you can open in any browser.
# include_plotlyjs=True embeds the library so it works offline. No nbformat / .show() needed.
fig = plot(top['Symbol'].iloc[0])
fig.write_html('outputs/chart.html', include_plotlyjs=True, auto_open=False)
print('saved outputs/chart.html')


saved outputs/chart.html


### Exit Points

In [19]:
if bottom['Sell'].any():
    print("Exit candidates trending down showing prices below Kalman:")
    for s in bottom.loc[bottom['Sell'], 'Symbol']: print(" ", s)
else: print("No short signals today; showing weakest-ranked names.")
bottom.style.map(color_boolean)

Exit candidates trending down showing prices below Kalman:
  BNB/USDT
  NIGHT/USDT


,Symbol,Buy,Sell,Close,RSI,Chop,AMAT,Ichimoku_9,Ichimoku_26,EMA_gt_Kalman,Low_gt_Kalman,Doji,Dragonfly,Gravestone,MACD,MACD_Signal,MACD_Buy,MACD_Sell,MACD_below_zero
0,BNB/USDT,False,True,601.630000,43.410000,55.320000,False,628.945000,651.100000,False,False,False,False,False,-10.923900,-11.429000,False,False,True
1,NIGHT/USDT,False,True,0.030300,42.570000,54.590000,False,0.033900,0.035200,False,False,False,False,False,-0.000600,-0.000400,False,False,True
2,WLD/USDT,True,False,0.658700,68.500000,54.270000,True,0.533600,0.474800,True,True,False,False,False,0.074500,0.059700,False,False,False
3,XPL/USDT,True,False,0.112700,67.440000,39.660000,True,0.091700,0.091700,True,True,False,False,False,0.001600,-0.002400,False,False,False
4,ASTER/USDT,False,False,0.717000,63.550000,36.000000,True,0.698500,0.695500,False,False,False,False,False,-0.002900,-0.009300,False,False,True
5,USD1/USDT,True,False,1.000800,62.700000,63.370000,True,1.000100,1.000000,True,True,False,False,False,0.000200,0.000100,False,False,False
6,UNI/USDT,False,False,3.227000,61.760000,30.810000,True,3.035000,3.243000,False,True,False,False,False,-0.069400,-0.170400,False,False,True
7,XLM/USDT,True,False,0.225600,61.690000,55.770000,True,0.210500,0.218900,True,True,False,False,False,0.007700,0.007300,True,False,False
8,FDUSD/USDT,False,False,0.999000,56.980000,45.300000,True,0.998300,0.998000,False,True,False,False,False,-0.000000,-0.000100,False,False,True
9,USDC/USDT,False,False,1.000600,52.780000,17.310000,True,1.008000,1.010800,False,False,False,False,False,-0.000000,-0.000000,False,False,True



### Four Votes
MACD is derived as guarded crossover points in the `macd_lab` metrics which is presented 
in terms of a regime going forward. That is, after a guarded buy the stance stays +1 until 
a guarded sell flips it. MA crossover is the plain fast-over-slow test from the MA-crossover notebook, +1 when the 20-period SMA is above the 50, -1 below. 

Fibonacci is contextual: on a rising leg, where price pulled back into the 0.5-0.618 golden pocket votes +1 to buy the dip; on a falling leg, price bounced into that pocket votes -1 (sell the rip); otherwise 0. Candles are an adjusted but assuming pattern, where +1 engulfs on a bullish and -1 on bearish, held live for a few bars before lapsing.

Worth noting that we detected a real bug in Candle module, a positional column rename that swapped open and close, so its pattern was informed by invalid fields. The version here 
uses the standard engulfing definition derived from valid columns.

### 2-3-Rule Thresholds
A threshold of 2 point agreement is recommended always. Raise it to 3 for stricter agreement and fewer trades. Everything is causal: each stance at bar i uses only bars up to i, so the backtest reflects what was knowable live. Fires are edge-triggered on the score crossing the threshold, so the dashboard shows more markers than the long-flat backtest executes, since a second buy while already long does not open a new position.

```
score = w_macd*MACD + w_ma*MA + w_fib*Fib + w_candle*Candle      (weights default 1)
BUY  fires when score first reaches +threshold   (default +2)
SELL fires when score first reaches -threshold   (default -2)
```

### Honest Cynicis Check
Hourly data, the ten-coin universe, about 1000 bars (~41 days), 0.1% fee per side,
long-flat, threshold 2. This window was a broad crypto downtrend.

| coin | strategy | buy & hold | trades | win % | max DD |
|---|---|---|---|---|---|
| BTC | -1.4% | -21.3% | 4 | 50 | -7.3% |
| ETH | -9.2% | -26.1% | 5 | 20 | -16.1% |
| SOL | -19.8% | -22.1% | 6 | 33 | -36.9% |
| BNB | -0.9% | -9.9% | 5 | 40 | -6.7% |
| XRP | -6.3% | -17.9% | 4 | 50 | -15.5% |
| ADA | -16.9% | -38.2% | 5 | 20 | -22.0% |
| AVAX | -8.0% | -33.6% | 6 | 33 | -16.7% |
| LINK | -19.7% | -20.3% | 7 | 29 | -25.4% |
| LTC | -10.1% | -23.4% | 4 | 25 | -17.1% |
| DOGE | -9.2% | -22.6% | 3 | 33 | -15.1% |
| mean | -10.1% | -23.5% | | | |

Read it plainly. The engine lost money on every coin, but lost roughly half of what
holding lost, because staying flat through the downtrend avoided the worst of it.
That is a drawdown-reduction property, not an edge. Beating buy-and-hold by being
absent during a fall is easy and does not survive into a rising or sideways market,
where sitting flat means missing gains. Win rates of 20 to 50 percent confirm there
is no demonstrated predictive skill here yet. This matches the project's standing
NO-GO finding: cleaner instrumentation, still no proven edge.

These are in-sample numbers over a single short, one-directional window. They tell
you the rule does something coherent, not that it will keep working.

### Change of Heart
Walk-forward is the next step the data has now earned: split each coin's history
into rolling train and test segments, choose the weights and threshold on train
only, score once on the untouched test segment, repeat forward, and report only the
out-of-sample aggregate with fees. Test across regimes, not just this falling one,
so a bull and a sideways stretch are included. Add a stop and a take-profit, since a
flat-long rule with no risk control flatters drawdown. Only if the out-of-sample,
multi-regime, after-fees result clearly beats both buy-and-hold and a coin-flip is
any of this worth real money, and even then the operator owns the decision and the
live switch stays off.

In [ ]:
stamp = datetime.now().strftime('%Y%m%d')
path = f'./outputs/DailySignals_{stamp}.csv'
results.to_csv(path, index=False)
print("saved", path)

saved ./outputs/DailySignals_20260618.csv


### Decision Checklist

1. Read balances (USDT, BTC, ...).
2. Size each trade as cash divided by the number of long candidates.
3. Act on exits (`bottom`) before entries (`top`).
4. Widen timelines and trend comparisons (all data saved & dated in `/outputs/`).


#### Render Reports using Quarto

 - `quarto preview day-metrics.ipynb --no-execute`
 - `quarto render day-metrics.ipynb --to html --no-execute`